**Lab 6 - Part 3**

# Task 1: Tải và Tiền xử lý Dữ liệu

In [1]:
# Kiểm tra đường dẫn file dữ liệu
import os
import conllu

# Thử các đường dẫn có thể có
possible_paths = [
    '../../data/lab6/UD_English-EWT/',
    '../../data/UD_English-EWT/',
    '../data/lab6/UD_English-EWT/',
    '../data/UD_English-EWT/',
    './data/lab6/UD_English-EWT/',
    './data/UD_English-EWT/'
]

folder_path = None
for path in possible_paths:
    if os.path.exists(path):
        folder_path = path
        print(f"Tìm thấy thư mục dữ liệu tại: {path}")
        break

if folder_path is None:
    print("Không tìm thấy thư mục dữ liệu. Hãy kiểm tra lại cấu trúc thư mục.")
    print("Thư mục hiện tại:", os.getcwd())
    print("Nội dung thư mục hiện tại:")
    for item in os.listdir('.'):
        print(f"  {item}")
else:
    train_file = folder_path + 'en_ewt-ud-train.conllu'
    dev_file = folder_path + 'en_ewt-ud-dev.conllu'
    test_file = folder_path + 'en_ewt-ud-test.conllu'
    
    # Kiểm tra file có tồn tại không
    for file_name, file_path in [('Train', train_file), ('Dev', dev_file), ('Test', test_file)]:
        if os.path.exists(file_path):
            print(f"✓ {file_name} file: {file_path}")
        else:
            print(f"✗ {file_name} file không tồn tại: {file_path}")


Tìm thấy thư mục dữ liệu tại: ../../data/UD_English-EWT/
✓ Train file: ../../data/UD_English-EWT/en_ewt-ud-train.conllu
✓ Dev file: ../../data/UD_English-EWT/en_ewt-ud-dev.conllu
✓ Test file: ../../data/UD_English-EWT/en_ewt-ud-test.conllu


In [2]:
def load_conllu(file_path):
    """
    Đọc dữ liệu từ file .conllu và trả về danh sách các câu
    
    Args:
        file_path (str): Đường dẫn đến file .conllu
        
    Returns:
        list: Danh sách các câu, mỗi câu là danh sách các cặp (word, upos_tag)
    """
    sentences = []
    
    with open(file_path, 'r', encoding='utf-8') as f:
        data = conllu.parse(f.read())
    
    for sentence in data:
        word_tag_pairs = []
        for token in sentence:
            # Bỏ qua các token có ID phức tạp (như 8-9 trong multiword tokens)
            if isinstance(token['id'], int):
                word = token['form']
                upos_tag = token['upos']
                word_tag_pairs.append((word, upos_tag))
        
        if word_tag_pairs:  # Chỉ thêm câu không rỗng
            sentences.append(word_tag_pairs)
    
    return sentences

# Test hàm load_conllu
print("Đang tải dữ liệu...")
train_sentences = load_conllu(train_file)
dev_sentences = load_conllu(dev_file)
test_sentences = load_conllu(test_file)

print(f"Số câu trong tập train: {len(train_sentences)}")
print(f"Số câu trong tập dev: {len(dev_sentences)}")
print(f"Số câu trong tập test: {len(test_sentences)}")

# Hiển thị một số ví dụ
print("\nVí dụ câu đầu tiên trong tập train:")
print(train_sentences[0][:10])  # Chỉ hiển thị 10 từ đầu tiên

Đang tải dữ liệu...
Số câu trong tập train: 12544
Số câu trong tập dev: 2001
Số câu trong tập test: 2077

Ví dụ câu đầu tiên trong tập train:
[('Al', 'PROPN'), ('-', 'PUNCT'), ('Zaman', 'PROPN'), (':', 'PUNCT'), ('American', 'ADJ'), ('forces', 'NOUN'), ('killed', 'VERB'), ('Shaikh', 'PROPN'), ('Abdullah', 'PROPN'), ('al', 'PROPN')]
Số câu trong tập train: 12544
Số câu trong tập dev: 2001
Số câu trong tập test: 2077

Ví dụ câu đầu tiên trong tập train:
[('Al', 'PROPN'), ('-', 'PUNCT'), ('Zaman', 'PROPN'), (':', 'PUNCT'), ('American', 'ADJ'), ('forces', 'NOUN'), ('killed', 'VERB'), ('Shaikh', 'PROPN'), ('Abdullah', 'PROPN'), ('al', 'PROPN')]


In [3]:
def build_vocabulary(train_sentences):
    """
    Xây dựng từ điển từ dữ liệu huấn luyện
    
    Args:
        train_sentences (list): Danh sách các câu từ tập train
        
    Returns:
        tuple: (word_to_ix, tag_to_ix) - hai từ điển ánh xạ
    """
    # Tập hợp tất cả các từ và tags
    all_words = set()
    all_tags = set()
    
    for sentence in train_sentences:
        for word, tag in sentence:
            all_words.add(word)
            all_tags.add(tag)
    
    # Tạo word_to_ix với <UNK> token đặc biệt
    word_to_ix = {'<UNK>': 0}
    for word in sorted(all_words):  # Sắp xếp để đảm bảo consistent indexing
        word_to_ix[word] = len(word_to_ix)
    
    # Tạo tag_to_ix
    tag_to_ix = {}
    for tag in sorted(all_tags):  # Sắp xếp để đảm bảo consistent indexing
        tag_to_ix[tag] = len(tag_to_ix)
    
    return word_to_ix, tag_to_ix

# Xây dựng từ điển từ dữ liệu train
print("Đang xây dựng từ điển...")
word_to_ix, tag_to_ix = build_vocabulary(train_sentences)

print(f"\nKích thước từ điển từ (word_to_ix): {len(word_to_ix)}")
print(f"Kích thước từ điển nhãn (tag_to_ix): {len(tag_to_ix)}")

print(f"\nCác nhãn UPOS có trong dữ liệu:")
for tag, idx in sorted(tag_to_ix.items(), key=lambda x: x[1]):
    print(f"  {tag}: {idx}")

print(f"\nMột số từ đầu tiên trong từ điển:")
for i, (word, idx) in enumerate(sorted(word_to_ix.items(), key=lambda x: x[1])[:10]):
    print(f"  {word}: {idx}")

print("...")
print(f"Tổng số từ (bao gồm <UNK>): {len(word_to_ix)}")

Đang xây dựng từ điển...

Kích thước từ điển từ (word_to_ix): 19674
Kích thước từ điển nhãn (tag_to_ix): 17

Các nhãn UPOS có trong dữ liệu:
  ADJ: 0
  ADP: 1
  ADV: 2
  AUX: 3
  CCONJ: 4
  DET: 5
  INTJ: 6
  NOUN: 7
  NUM: 8
  PART: 9
  PRON: 10
  PROPN: 11
  PUNCT: 12
  SCONJ: 13
  SYM: 14
  VERB: 15
  X: 16

Một số từ đầu tiên trong từ điển:
  <UNK>: 0
  !: 1
  !!: 2
  !!!: 3
  !!!!: 4
  !!!!!: 5
  !!!!!!: 6
  !!!!!!!: 7
  !!!!!!!!!!: 8
  !!!!!!!!!!!: 9
...
Tổng số từ (bao gồm <UNK>): 19674


In [4]:
# Hash-based vocabulary tương tự BigDL Scala code
import hashlib

def hash_word(word, vocab_size=16384):
    """
    Hash một từ thành index sử dụng hash function tương tự Murmur3
    
    Args:
        word (str): Từ cần hash
        vocab_size (int): Kích thước vocabulary
        
    Returns:
        int: Hash index từ 1 đến vocab_size (0 reserved cho padding)
    """
    # Sử dụng hashlib để tạo hash tương tự Murmur3
    hash_object = hashlib.md5(word.encode('utf-8'))
    hash_hex = hash_object.hexdigest()
    hash_int = int(hash_hex, 16)
    
    # Modulo để đảm bảo trong khoảng vocab_size, +1 để tránh index 0 (reserved cho padding)
    return (hash_int % (vocab_size - 1)) + 1

def build_vocabulary_hash_based(train_sentences, vocab_size=16384):
    """
    Xây dựng hash-based vocabulary tương tự BigDL
    
    Args:
        train_sentences (list): Danh sách các câu từ tập train
        vocab_size (int): Kích thước vocabulary (default: 16384 như Scala code)
        
    Returns:
        tuple: (hash_word_func, tag_to_ix) - hàm hash và từ điển tags
    """
    # Tập hợp tất cả các tags
    all_tags = set()
    
    for sentence in train_sentences:
        for word, tag in sentence:
            all_tags.add(tag)
    
    # Tạo tag_to_ix (giữ nguyên như trước, bắt đầu từ 0)
    tag_to_ix = {}
    for tag in sorted(all_tags):
        tag_to_ix[tag] = len(tag_to_ix)
    
    # Tạo hàm hash cho từ
    def hash_word_func(word):
        return hash_word(word, vocab_size)
    
    return hash_word_func, tag_to_ix, vocab_size

# Test hash-based vocabulary
print("Xây dựng hash-based vocabulary...")
hash_word_func, tag_to_ix_hash, VOCAB_SIZE_HASH = build_vocabulary_hash_based(train_sentences, vocab_size=16384)

print(f"\nHash-based vocabulary:")
print(f"  - Vocabulary size: {VOCAB_SIZE_HASH}")
print(f"  - Tag dictionary size: {len(tag_to_ix_hash)}")

# Test hash function với một số từ
test_words = ["the", "quick", "brown", "fox", "jumps", "over", "lazy", "dog"]
print(f"\nTest hash function:")
for word in test_words:
    hash_idx = hash_word_func(word)
    print(f"  '{word}' -> {hash_idx}")

# Kiểm tra tính nhất quán của hash
print(f"\nTính nhất quán của hash function:")
for word in test_words[:3]:
    hash1 = hash_word_func(word)
    hash2 = hash_word_func(word)
    print(f"  '{word}': {hash1} == {hash2} -> {hash1 == hash2}")

print(f"\nCác nhãn UPOS (hash-based):")
for tag, idx in sorted(tag_to_ix_hash.items(), key=lambda x: x[1]):
    print(f"  {tag}: {idx}")

Xây dựng hash-based vocabulary...

Hash-based vocabulary:
  - Vocabulary size: 16384
  - Tag dictionary size: 17

Test hash function:
  'the' -> 6087
  'quick' -> 4399
  'brown' -> 3508
  'fox' -> 5816
  'jumps' -> 16315
  'over' -> 15485
  'lazy' -> 14630
  'dog' -> 9240

Tính nhất quán của hash function:
  'the': 6087 == 6087 -> True
  'quick': 4399 == 4399 -> True
  'brown': 3508 == 3508 -> True

Các nhãn UPOS (hash-based):
  ADJ: 0
  ADP: 1
  ADV: 2
  AUX: 3
  CCONJ: 4
  DET: 5
  INTJ: 6
  NOUN: 7
  NUM: 8
  PART: 9
  PRON: 10
  PROPN: 11
  PUNCT: 12
  SCONJ: 13
  SYM: 14
  VERB: 15
  X: 16


In [5]:
# Fixed sequence length processing tương tự BigDL (maxSeqLen = 30)
MAX_SEQ_LEN = 30

def sentence_to_fixed_indices(sentence, hash_word_func, tag_to_ix, max_seq_len=MAX_SEQ_LEN):
    """
    Chuyển đổi một câu thành fixed-length indices tương tự BigDL
    
    Args:
        sentence (list): Danh sách các cặp (word, tag)
        hash_word_func: Hàm hash cho từ
        tag_to_ix (dict): Từ điển ánh xạ tag -> index
        max_seq_len (int): Độ dài sequence cố định
    
    Returns:
        tuple: (word_indices, tag_indices) - cả hai đều có length = max_seq_len
    """
    word_indices = []
    tag_indices = []
    
    # Xử lý từng từ trong câu
    for word, tag in sentence:
        if len(word_indices) >= max_seq_len:
            break  # Cắt nếu vượt quá độ dài tối đa
            
        word_idx = hash_word_func(word)
        tag_idx = tag_to_ix[tag]
        
        word_indices.append(word_idx)
        tag_indices.append(tag_idx)
    
    # Padding với 0 cho words và -1 cho tags (như BigDL)
    if len(word_indices) < max_seq_len:
        word_indices.extend([0] * (max_seq_len - len(word_indices)))
        tag_indices.extend([-1] * (max_seq_len - len(tag_indices)))
    
    return word_indices, tag_indices

# Test với câu ví dụ
example_sentence = train_sentences[0]
print(f"Câu ví dụ gốc (length={len(example_sentence)}):")
print(example_sentence[:10])  # 10 từ đầu

word_ids_hash, tag_ids_hash = sentence_to_fixed_indices(example_sentence, hash_word_func, tag_to_ix_hash)
print(f"\nFixed-length indices (length={len(word_ids_hash)}):")
print(f"Word indices (first 15): {word_ids_hash[:15]}")
print(f"Tag indices (first 15):  {tag_ids_hash[:15]}")

print(f"\nPadding examples:")
print(f"Word indices (last 10): {word_ids_hash[-10:]}")
print(f"Tag indices (last 10):  {tag_ids_hash[-10:]}")

# Test với câu ngắn
short_sentence = [("Hello", "INTJ"), ("world", "NOUN")]
word_ids_short, tag_ids_short = sentence_to_fixed_indices(short_sentence, hash_word_func, tag_to_ix_hash)
print(f"\nCâu ngắn test:")
print(f"Original: {short_sentence}")
print(f"Word indices: {word_ids_short}")
print(f"Tag indices: {tag_ids_short}")

Câu ví dụ gốc (length=29):
[('Al', 'PROPN'), ('-', 'PUNCT'), ('Zaman', 'PROPN'), (':', 'PUNCT'), ('American', 'ADJ'), ('forces', 'NOUN'), ('killed', 'VERB'), ('Shaikh', 'PROPN'), ('Abdullah', 'PROPN'), ('al', 'PROPN')]

Fixed-length indices (length=30):
Word indices (first 15): [6681, 11715, 4541, 16279, 3509, 197, 410, 5709, 4642, 1585, 11715, 207, 13427, 6087, 14243]
Tag indices (first 15):  [11, 12, 11, 12, 0, 7, 15, 11, 11, 11, 12, 11, 12, 5, 7]

Padding examples:
Word indices (last 10): [13451, 246, 3848, 13427, 5801, 6087, 15717, 12972, 12034, 0]
Tag indices (last 10):  [7, 1, 11, 12, 1, 5, 0, 7, 12, -1]

Câu ngắn test:
Original: [('Hello', 'INTJ'), ('world', 'NOUN')]
Word indices: [10457, 7830, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Tag indices: [6, 7, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1]


In [6]:
# Tạo hàm helper để chuyển đổi câu thành indices
def sentence_to_indices(sentence, word_to_ix, tag_to_ix):
    """
    Chuyển đổi một câu thành danh sách các indices
    
    Args:
        sentence (list): Danh sách các cặp (word, tag)
        word_to_ix (dict): Từ điển ánh xạ từ -> index
        tag_to_ix (dict): Từ điển ánh xạ tag -> index
    
    Returns:
        tuple: (word_indices, tag_indices)
    """
    word_indices = []
    tag_indices = []
    
    for word, tag in sentence:
        # Sử dụng <UNK> nếu từ không có trong từ điển
        word_idx = word_to_ix.get(word, word_to_ix['<UNK>'])
        tag_idx = tag_to_ix[tag]
        
        word_indices.append(word_idx)
        tag_indices.append(tag_idx)
    
    return word_indices, tag_indices

# Test với một câu ví dụ
example_sentence = train_sentences[0]
print("Câu ví dụ (5 từ đầu):")
print(example_sentence[:5])

word_ids, tag_ids = sentence_to_indices(example_sentence, word_to_ix, tag_to_ix)
print("\nWord indices (5 đầu):")
print(word_ids[:5])
print("\nTag indices (5 đầu):")
print(tag_ids[:5])

# Kiểm tra reverse mapping
print("\nKiểm tra reverse mapping:")
ix_to_word = {idx: word for word, idx in word_to_ix.items()}
ix_to_tag = {idx: tag for tag, idx in tag_to_ix.items()}

for i in range(min(5, len(word_ids))):
    original_word, original_tag = example_sentence[i]
    reconstructed_word = ix_to_word[word_ids[i]]
    reconstructed_tag = ix_to_tag[tag_ids[i]]
    print(f"  '{original_word}'/'{original_tag}' -> {word_ids[i]}/{tag_ids[i]} -> '{reconstructed_word}'/'{reconstructed_tag}'")

Câu ví dụ (5 từ đầu):
[('Al', 'PROPN'), ('-', 'PUNCT'), ('Zaman', 'PROPN'), (':', 'PUNCT'), ('American', 'ADJ')]

Word indices (5 đầu):
[1361, 81, 7684, 1127, 1416]

Tag indices (5 đầu):
[11, 12, 11, 12, 0]

Kiểm tra reverse mapping:
  'Al'/'PROPN' -> 1361/11 -> 'Al'/'PROPN'
  '-'/'PUNCT' -> 81/12 -> '-'/'PUNCT'
  'Zaman'/'PROPN' -> 7684/11 -> 'Zaman'/'PROPN'
  ':'/'PUNCT' -> 1127/12 -> ':'/'PUNCT'
  'American'/'ADJ' -> 1416/0 -> 'American'/'ADJ'


In [7]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from transformers import BertTokenizer, BertModel
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from tqdm import tqdm   

C:\Users\Admin\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Task 2: Tạo PyTorch Dataset và DataLoader

In [8]:
class POSDataset(Dataset):
    """
    PyTorch Dataset cho bài toán POS tagging với hash-based vocabulary (tương tự BigDL)
    """
    
    def __init__(self, sentences, hash_word_func, tag_to_ix, max_seq_len=MAX_SEQ_LEN):
        """
        Khởi tạo dataset với hash-based approach
        
        Args:
            sentences (list): Danh sách các câu, mỗi câu là list các cặp (word, tag)
            hash_word_func: Hàm hash cho từ
            tag_to_ix (dict): Từ điển ánh xạ tag -> index
            max_seq_len (int): Độ dài sequence cố định
        """
        self.sentences = sentences
        self.hash_word_func = hash_word_func
        self.tag_to_ix = tag_to_ix
        self.max_seq_len = max_seq_len
    
    def __len__(self):
        """
        Trả về tổng số câu trong dataset
        """
        return len(self.sentences)
    
    def __getitem__(self, idx):
        """
        Lấy một mẫu dữ liệu tại index idx
        
        Args:
            idx (int): Index của câu cần lấy
            
        Returns:
            tuple: (sentence_tensor, tag_tensor) - cả hai đều có shape (max_seq_len,)
        """
        sentence = self.sentences[idx]
        
        # Chuyển đổi câu thành fixed-length indices
        sentence_indices, tag_indices = sentence_to_fixed_indices(
            sentence, self.hash_word_func, self.tag_to_ix, self.max_seq_len
        )
        
        # Chuyển thành tensors với fixed shape
        sentence_tensor = torch.tensor(sentence_indices, dtype=torch.long)
        tag_tensor = torch.tensor(tag_indices, dtype=torch.long)
        
        return sentence_tensor, tag_tensor

# Test lớp POSDataset
print("Tạo hash-based datasets...")
train_dataset_hash = POSDataset(train_sentences, hash_word_func, tag_to_ix_hash)
dev_dataset_hash = POSDataset(dev_sentences, hash_word_func, tag_to_ix_hash)
test_dataset_hash = POSDataset(test_sentences, hash_word_func, tag_to_ix_hash)

print(f"Hash-based datasets:")
print(f"  - Train dataset size: {len(train_dataset_hash)}")
print(f"  - Dev dataset size: {len(dev_dataset_hash)}")
print(f"  - Test dataset size: {len(test_dataset_hash)}")

# Test __getitem__
sample_idx = 0
sentence_tensor, tag_tensor = train_dataset_hash[sample_idx]
print(f"\nMẫu thứ {sample_idx} (fixed-length):")
print(f"  - Sentence tensor shape: {sentence_tensor.shape}")
print(f"  - Tag tensor shape: {tag_tensor.shape}")
print(f"  - Sentence indices (first 15): {sentence_tensor[:15]}")
print(f"  - Tag indices (first 15): {tag_tensor[:15]}")
print(f"  - Sentence indices (last 10): {sentence_tensor[-10:]}")  # Kiểm tra padding
print(f"  - Tag indices (last 10): {tag_tensor[-10:]}")  # Kiểm tra padding

Tạo hash-based datasets...
Hash-based datasets:
  - Train dataset size: 12544
  - Dev dataset size: 2001
  - Test dataset size: 2077

Mẫu thứ 0 (fixed-length):
  - Sentence tensor shape: torch.Size([30])
  - Tag tensor shape: torch.Size([30])
  - Sentence indices (first 15): tensor([ 6681, 11715,  4541, 16279,  3509,   197,   410,  5709,  4642,  1585,
        11715,   207, 13427,  6087, 14243])
  - Tag indices (first 15): tensor([11, 12, 11, 12,  0,  7, 15, 11, 11, 11, 12, 11, 12,  5,  7])
  - Sentence indices (last 10): tensor([13451,   246,  3848, 13427,  5801,  6087, 15717, 12972, 12034,     0])
  - Tag indices (last 10): tensor([ 7,  1, 11, 12,  1,  5,  0,  7, 12, -1])


In [ ]:
def simple_collate_fn(batch):
    """
    Simple collate function cho fixed-length sequences
    Không cần padding vì tất cả sequences đã có cùng độ dài
    """
    sentences, tags = zip(*batch)
    
    # Stack tensors trực tiếp (không cần padding)
    sentence_batch = torch.stack(sentences)
    tag_batch = torch.stack(tags)
    
    return sentence_batch, tag_batch

# Tạo DataLoaders với hash-based dataset
batch_size = 32

train_loader = DataLoader(
    train_dataset_hash, 
    batch_size=batch_size, 
    shuffle=True, 
    collate_fn=simple_collate_fn
)

dev_loader = DataLoader(
    dev_dataset_hash, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=simple_collate_fn
)

test_loader = DataLoader(
    test_dataset_hash, 
    batch_size=batch_size, 
    shuffle=False, 
    collate_fn=simple_collate_fn
)

print(f"Hash-based DataLoaders:")
print(f"  - Train batches: {len(train_loader)}")
print(f"  - Dev batches: {len(dev_loader)}")
print(f"  - Test batches: {len(test_loader  )}")

# Test một batch
print("\nTest batch từ hash-based train_loader:")
for batch_idx, (sentences, tags) in enumerate(train_loader):
    print(f"  - Batch shape - Sentences: {sentences.shape}, Tags: {tags.shape}")
    print(f"  - Sentences sample (first 5 of first sample): {sentences[0][:5]}")
    print(f"  - Tags sample (first 5 of first sample): {tags[0][:5]}")
    print(f"  - Check padding (last 5 of first sample):")
    print(f"    * Sentences: {sentences[0][-5:]}")
    print(f"    * Tags: {tags[0][-5:]}")
    break  # Chỉ test batch đầu tiên

Hash-based DataLoaders:
  - Train batches: 392
  - Dev batches: 63
  - Test batches: 65

Test batch từ hash-based train_loader:
  - Batch shape - Sentences: torch.Size([32, 30]), Tags: torch.Size([32, 30])
  - Sentences sample (first 5 of first sample): tensor([ 6684,  9588,  2922, 14198,  6290])
  - Tags sample (first 5 of first sample): tensor([5, 7, 7, 3, 0])
  - Check padding (last 5 of first sample):
    * Sentences: tensor([0, 0, 0, 0, 0])
    * Tags: tensor([-1, -1, -1, -1, -1])


# Task 3: Xây dựng Mô hình RNN

In [10]:
class SimpleRNNForTokenClassification(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_tags, num_layers=1, dropout=0.1):
        super(SimpleRNNForTokenClassification, self).__init__()
        
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.num_tags = num_tags
        self.num_layers = num_layers
        
        # 1. Embedding layer: (batch_size, seq_len) -> (batch_size, seq_len, embedding_dim)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # 2. RNN layer: (batch_size, seq_len, embedding_dim) -> (batch_size, seq_len, hidden_dim)
        self.rnn = nn.RNN(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=False
        )
        
        # 3. Dropout for regularization
        self.dropout_layer = nn.Dropout(dropout)
        
        # 4. Linear classifier: (batch_size, seq_len, hidden_dim) -> (batch_size, seq_len, num_tags)
        self.classifier = nn.Linear(hidden_dim, num_tags)
        
        self._init_weights()
    
    def _init_weights(self):
        nn.init.normal_(self.embedding.weight, mean=0.0, std=0.1)
        for name, param in self.rnn.named_parameters():
            if 'weight' in name:
                nn.init.xavier_uniform_(param)
            elif 'bias' in name:
                nn.init.constant_(param, 0)
        nn.init.xavier_uniform_(self.classifier.weight)
        nn.init.constant_(self.classifier.bias, 0)
    
    def forward(self, input_ids, lengths=None):
        embeddings = self.embedding(input_ids)
        rnn_output, hidden = self.rnn(embeddings)
        rnn_output = self.dropout_layer(rnn_output)
        logits = self.classifier(rnn_output)
        return logits
    
    def init_hidden(self, batch_size, device):
        return torch.zeros(self.num_layers, batch_size, self.hidden_dim, device=device)

# Test model - using hash-based vocabulary for consistency
VOCAB_SIZE = VOCAB_SIZE_HASH  # 16384 to match hash-based DataLoader
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
NUM_TAGS = len(tag_to_ix_hash)  # Use hash-based tags

model_simple = SimpleRNNForTokenClassification(
    vocab_size=VOCAB_SIZE,
    embedding_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_tags=NUM_TAGS
)

total_params = sum(p.numel() for p in model_simple.parameters())
print(f"Model created: {total_params:,} parameters")

# Quick test
test_input = torch.randint(0, VOCAB_SIZE, (2, 5))
model_simple.eval()
with torch.no_grad():
    output = model_simple(test_input)
    print(f"Test passed: {test_input.shape} -> {output.shape}")

Model created: 1,670,033 parameters
Test passed: torch.Size([2, 5]) -> torch.Size([2, 5, 17])


# Task 4: Huấn luyện mô hình

In [ ]:
# Setup training components
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Use the existing SimpleRNN model
model = model_simple.to(device)
print(f"Using SimpleRNNForTokenClassification")

# Training parameters
learning_rate = 0.001
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss(ignore_index=-1)
num_epochs = 30

def train_model(model, train_loader, dev_loader, optimizer, criterion, device, num_epochs=30):
    """
    Training function for POS tagging model
    """
    # TensorBoard setup
    try:
        from torch.utils.tensorboard import SummaryWriter
        import datetime
        
        current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
        log_dir = f"runs/simple_pos_tagger_{current_time}"
        writer = SummaryWriter(log_dir)
        print(f"TensorBoard logging: {log_dir}")
        tensorboard_available = True
    except ImportError:
        print("TensorBoard not available")
        writer = None
        tensorboard_available = False

    print(f"Training setup:")
    print(f"  - Epochs: {num_epochs}")
    print(f"  - Optimizer: Adam")
    print(f"  - TensorBoard: {'Enabled' if tensorboard_available else 'Disabled'}")

    # Training loop
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params:,}")

    print("\nStarting training...")
    model.train()

    global_step = 0
    for epoch in range(num_epochs):
        epoch_loss = 0
        for batch_idx, (sentences, tags) in enumerate(train_loader):
            sentences, tags = sentences.to(device), tags.to(device)
            
            optimizer.zero_grad()
            logits = model(sentences)
            
            # Flatten for loss computation
            batch_size, seq_len, num_tags = logits.shape
            logits_flat = logits.view(batch_size * seq_len, num_tags)
            tags_flat = tags.view(batch_size * seq_len)
            
            loss = criterion(logits_flat, tags_flat)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            
            # Log to TensorBoard
            if tensorboard_available and global_step % 50 == 0:
                writer.add_scalar('Loss/Train', loss.item(), global_step)
            
            global_step += 1
        
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {avg_loss:.4f}")
        
        # Log epoch metrics to TensorBoard
        if tensorboard_available:
            writer.add_scalar('Loss/Epoch', avg_loss, epoch)
            
            # Evaluate on dev set
            dev_acc, dev_loss = evaluate_model(model, dev_loader, device, criterion)
            writer.add_scalar('Accuracy/Dev', dev_acc, epoch)
            writer.add_scalar('Loss/Dev', dev_loss, epoch)
            print(f"  Dev Accuracy: {dev_acc:.4f}, Dev Loss: {dev_loss:.4f}")

    print("Training completed!")

    # Save model
    torch.save(model.state_dict(), 'pos_tagger_model.pth')
    print("Model saved!")

    # Close TensorBoard writer
    if tensorboard_available:
        writer.close()
        print(f"TensorBoard logs saved to: {log_dir}")
        print("To view: tensorboard --logdir=runs")
    
    return model

Device: cpu
Using SimpleRNNForTokenClassification


# Task 5: Đánh giá Mô hình

In [17]:
# Simple evaluation function
def evaluate_model(model, dataloader, device, criterion):
    """
    Evaluate model on given dataloader
    """
    model.eval()
    total_correct = 0
    total_tokens = 0
    total_loss = 0
    total_batches = 0
    
    with torch.no_grad():
        for sentences, tags in dataloader:
            sentences, tags = sentences.to(device), tags.to(device)
            logits = model(sentences)
            predictions = torch.argmax(logits, dim=-1)
            
            # Loss
            batch_size, seq_len, num_tags = logits.shape
            logits_flat = logits.view(batch_size * seq_len, num_tags)
            tags_flat = tags.view(batch_size * seq_len)
            loss = criterion(logits_flat, tags_flat)
            total_loss += loss.item()
            total_batches += 1
            
            # Accuracy (exclude padding tokens)
            mask = (tags != -1)
            correct = ((predictions == tags) & mask).sum().item()
            total_correct += correct
            total_tokens += mask.sum().item()
    
    accuracy = total_correct / total_tokens if total_tokens > 0 else 0
    avg_loss = total_loss / total_batches if total_batches > 0 else 0
    model.train()
    return accuracy, avg_loss

In [18]:
# Mở TensorBoard trong notebook
# Lưu ý: Chạy cell này TRONG QUÁ TRÌNH hoặc SAU KHI huấn luyện để xem real-time metrics

# Cách 1: Sử dụng magic command (khuyến nghị)
#%load_ext tensorboard
#%tensorboard --logdir=runs

# Cách 2: Nếu magic command không hoạt động, sử dụng code này:
# import subprocess
# subprocess.Popen(['tensorboard', '--logdir=runs', '--port=6006'])
# print("TensorBoard đã khởi động tại: http://localhost:6006")

# Cách 3: Inline TensorBoard widget (nếu có Jupyter extension)
# Mở TensorBoard trong notebook bằng inline widget
from tensorboard import notebook
notebook.display(port=6006, height=800)


Selecting TensorBoard with logdir runs (started 12:09:15 ago; port 6006, pid 19300).


In [ ]:
# Train and evaluate the model
print("=== TRAINING POS TAGGER ===")
trained_model = train_model(model, train_loader, dev_loader, optimizer, criterion, device, num_epochs)

print("\n=== FINAL EVALUATION ===")
# Final evaluation on test set
test_acc, test_loss = evaluate_model(trained_model, test_loader, device, criterion)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

# Final evaluation on dev set
dev_acc, dev_loss = evaluate_model(trained_model, dev_loader, device, criterion)
print(f"Dev Accuracy: {dev_acc:.4f}")
print(f"Dev Loss: {dev_loss:.4f}")

=== TRAINING POS TAGGER ===
TensorBoard logging: runs/simple_pos_tagger_20251118-000555
Training setup:
  - Epochs: 5
  - Optimizer: Adam
  - TensorBoard: Enabled
Total parameters: 1,670,033

Starting training...
Epoch 1/5, Loss: 0.1707
Epoch 1/5, Loss: 0.1707
  Dev Accuracy: 0.8472, Dev Loss: 0.6007
  Dev Accuracy: 0.8472, Dev Loss: 0.6007
Epoch 2/5, Loss: 0.1424
Epoch 2/5, Loss: 0.1424
  Dev Accuracy: 0.8455, Dev Loss: 0.6655
  Dev Accuracy: 0.8455, Dev Loss: 0.6655
Epoch 3/5, Loss: 0.1217
Epoch 3/5, Loss: 0.1217
  Dev Accuracy: 0.8421, Dev Loss: 0.7025
  Dev Accuracy: 0.8421, Dev Loss: 0.7025
Epoch 4/5, Loss: 0.1032
Epoch 4/5, Loss: 0.1032
  Dev Accuracy: 0.8389, Dev Loss: 0.7662
  Dev Accuracy: 0.8389, Dev Loss: 0.7662
